# Data Ingestion

**Project:** Airbnb Data Engineering Assessment  
**Goal:** Download London Airbnb data and load into DuckDB

**Why London?**
- ~80,000 listings (large enough for statistical analysis)
- English-language reviews
- Well-documented dataset
- 100+ neighborhoods for geographic analysis


In [11]:
import duckdb
import pandas as pd
import os

print("Libraries loaded successfully")
print(f"Working directory: {os.getcwd()}")
print(f"DuckDB version: {duckdb.__version__}")

Libraries loaded successfully
Working directory: g:\airbnb-data-engineering-assessment\notebooks
DuckDB version: 1.5.4


In [12]:
import urllib.request

os.makedirs('../data/raw', exist_ok=True)

urls = {
    'listings.csv': 'http://data.insideairbnb.com/united-kingdom/england/london/2024-09-06/visualisations/listings.csv',
    'reviews.csv': 'http://data.insideairbnb.com/united-kingdom/england/london/2024-09-06/visualisations/reviews.csv',
}

for filename, url in urls.items():
    filepath = f'../data/raw/{filename}'
    if not os.path.exists(filepath):
        print(f"Downloading {filename}...")
        urllib.request.urlretrieve(url, filepath)
        size_mb = os.path.getsize(filepath) / 1024 / 1024
        print(f"Done ({size_mb:.1f} MB)")
    else:
        print(f"{filename} already exists")

Done (13.7 MB)
Done (40.8 MB)


In [15]:
# Check downloaded files

import os
for f in os.listdir('../data/raw'):
    size_mb = os.path.getsize(f'../data/raw/{f}') / 1024 / 1024
    print(f"  📄 {f}: {size_mb:.1f} MB")

  📄 listings.csv: 13.7 MB
  📄 reviews.csv: 40.8 MB


In [16]:
# Load CSVs into DuckDB

# Connect to a database file (will be created if doesn't exist)
con = duckdb.connect('../data/airbnb.db')
print("Connected to DuckDB database!")

# Load listings.csv into a DuckDB table
con.execute("""
    CREATE OR REPLACE TABLE listings AS 
    SELECT * FROM read_csv_auto('../data/raw/listings.csv')
""")
print("Listings table created!")

# Load reviews.csv into a DuckDB table
con.execute("""
    CREATE OR REPLACE TABLE reviews AS 
    SELECT * FROM read_csv_auto('../data/raw/reviews.csv')
""")
print("Reviews table created!")


Connected to DuckDB database!
Listings table created!
Reviews table created!


### Count listings and reviews

In [17]:
result = con.execute("SELECT COUNT(*) as total FROM listings").fetchone()
print(f" Total listings: {result[0]:,}")

result = con.execute("SELECT COUNT(*) as total FROM reviews").fetchone()
print(f" Total reviews: {result[0]:,}")

 Total listings: 96,182
 Total reviews: 1,887,519


### Show listings schema

In [18]:
con.execute("DESCRIBE listings").df()

,column_name,column_type,null,key,default,extra
0,id,BIGINT,YES,None,None,None
1,name,VARCHAR,YES,None,None,None
2,host_id,BIGINT,YES,None,None,None
3,host_name,VARCHAR,YES,None,None,None
4,neighbourhood_group,VARCHAR,YES,None,None,None
5,neighbourhood,VARCHAR,YES,None,None,None
6,latitude,DOUBLE,YES,None,None,None
7,longitude,DOUBLE,YES,None,None,None
8,room_type,VARCHAR,YES,None,None,None
9,price,BIGINT,YES,None,None,None


### Peek at data

In [19]:
con.execute("SELECT * FROM listings LIMIT 3").df()

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365,number_of_reviews_ltm,license
0,116268,Double Room (Unavailable for check in 31Dec-1Jan),586671,Joe,None,Enfield,51.611370,-0.118880,Private room,40,7,38,2024-01-01,0.28,1,105,1,None
1,117203,A stylish Victorian home in West London,255103,Olga,None,Hammersmith and Fulham,51.501550,-0.233002,Entire home/apt,131,5,91,2024-06-09,0.59,1,33,10,None
2,127652,Contemporary central London apt,134938,Ron,None,Camden,51.559528,-0.144319,Entire home/apt,215,5,216,2024-07-09,1.35,1,134,5,None
